# research_report（研报盈利预测）全年推理结果分析

本 notebook 分析**2024 全年**研报推理结果（`artifacts/research_report_probs_forecast_stk_20240101_20241231.csv`，
单个文件，覆盖 `2024-01-01 ~ 2024-12-31`）。
推理由 `scripts/infer_research_report.py` 完成（文本 = `TITLE + 。 + CONTENT`，`max_length=512`，单卡 fp16），
本册只读产物做分析：逐月概况 / 概率分布 / 日度-个股聚合 / report_type·reliability 交叉 / 可视化。

**契约提醒（模型输出未确认项）：**
- 标签 `class_0/1` 语义未确认，**不要**命名为正面/负面/看多/看空；
- pooling 未确认（候选 `cls / pooler / masked_mean`），默认 `cls`（本册未重跑对比）；
- 结果为 fp16，与 fp32 概率约有 ~1e-3 差异；
- `report_type / reliability / organ_name / symbol` 为研报结构化字段，本册只做分布与交叉观察，**不臆断**其业务含义。

**与 social_text 分析的差异：**
- 本结果是**单个 CSV**（非 12 个月多文件 concat），同时保留自然发布日期 `date` 与市场可用日 `available_date`；
- 日度聚合使用 `available_date`：交易日 14:57（含）以前归当天，其后及非交易日归下一交易日；
- 额外提供 `report_type / reliability / organ_name` 的结构化字段交叉分析。

In [ ]:
# 环境与导入
%matplotlib inline
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
if not (ROOT / "configs" / "model.yaml").is_file():
    raise FileNotFoundError("请从仓库根目录启动 notebook，或设置 PROJECT_ROOT")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / "artifacts"

# 中文字体(避免 DejaVu 缺字/乱码)
plt.rcParams["font.sans-serif"] = ["WenQuanYi Micro Hei", "WenQuanYi Zen Hei", "AR PL UMing CN", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("python:", sys.version.split()[0], "| numpy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
# 加载全年研报推理结果(单个 CSV)
CSV = OUT_DIR / "research_report_probs_forecast_stk_20240101_20241231.csv"
res = pd.read_csv(CSV, dtype={"class_0_prob": np.float32, "class_1_prob": np.float32,
                              "id": str, "symbol": str})
required = {"date", "available_date", "class_0_prob", "class_1_prob", "symbol"}
missing = required - set(res.columns)
if missing:
    raise ValueError(f"推理结果缺少列: {sorted(missing)}")
res["date"] = pd.to_datetime(res["date"], errors="coerce")
res["available_date"] = pd.to_datetime(res["available_date"], errors="coerce")
res["month"] = res["available_date"].dt.strftime("%Y-%m")

print("加载完成: shape =", res.shape, "| 内存 ~%.1f MB" % (res.memory_usage(deep=True).sum()/1e6))
print("列:", list(res.columns))
res.head(3)

In [ ]:
# 结果体检: 行数 / NaN / 概率和 / 来源 / report_type / reliability / organ / 日期
print("逐月行数:")
print(res.groupby("month").size().rename("n_reports").to_string())
print()
print("NaN 数(class_0/1):", int(res[["class_0_prob", "class_1_prob"]].isna().sum().sum()))
print("reliability 缺失: %d (%.1f%%)" % (int(res["reliability"].isna().sum()), res["reliability"].isna().mean()*100))
s = res["class_0_prob"] + res["class_1_prob"]
print("概率和 max|sum-1|:", round(abs(s - 1).max(), 6))
print()
print("来源分布:")
print(res["source"].value_counts().to_string())
print()
print("report_type 分布(代码, 语义未确认):")
print(res["report_type"].value_counts().to_string())
print()
print("reliability 分布(大量缺失):")
print(res["reliability"].value_counts(dropna=False).to_string())
print()
print("organ_name: %d 家机构 | symbol: %d 只个股 | id 唯一: %d" % (res["organ_name"].nunique(), res["symbol"].nunique(), res["id"].nunique()))
print("organ_name top 10:")
print(res["organ_name"].astype(str).value_counts().head(10).to_string())
print()
print("自然发布日期覆盖: %s -> %s | 唯一日期 %d" % (res["date"].min(), res["date"].max(), res["date"].nunique()))
print("市场可用日覆盖: %s -> %s | 交易日 %d" % (res["available_date"].min(), res["available_date"].max(), res["available_date"].nunique()))
print("星期分布(0=周一..6=周日, 研报含周末发布):")
print(res["date"].dt.dayofweek.value_counts().sort_index().to_string())

In [ ]:
# class_1_prob 分布: 整体 + 逐月
print("class_1_prob 整体分布:")
print(res["class_1_prob"].describe().round(4).to_string())
print()
mm = res.groupby("month")["class_1_prob"].agg(["mean", "median", "std", "count"])
print("逐月 class_1_prob:")
print(mm.round(4).to_string())

In [ ]:
# 可视化: 直方图 / 逐月均值 / 样本个股日度时序
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# 1) 直方图(研报量 ~27 万, 直接全量)
axes[0].hist(res["class_1_prob"], bins=60)
axes[0].set_title("class_1_prob 分布")
axes[0].set_xlabel("class_1_prob")

# 2) 逐月 class_1_prob 均值
mm = res.groupby("month")["class_1_prob"].mean()
axes[1].plot(mm.index, mm.values, marker="o")
axes[1].set_title("逐月 class_1_prob 均值")
axes[1].tick_params(axis="x", rotation=45, labelsize=8)

# 3) 样本个股日度均值时序
top_sym = res["symbol"].value_counts().index[0]
d = (res[res["symbol"] == top_sym]
     .groupby("available_date")["class_1_prob"].mean().sort_index())
axes[2].plot(d.index.astype(str), d.values, lw=0.8)
axes[2].set_title(f"{top_sym} 日度 class_1_prob 均值")
axes[2].tick_params(axis="x", rotation=90, labelsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# 日度-个股情绪分数：按市场可用交易日聚合
daily = (res.groupby(["available_date", "symbol"])["class_1_prob"]
         .agg(["mean", "median", "std", "count"]).reset_index())
daily.columns = ["available_date", "symbol", "sentiment_mean", "sentiment_median", "sentiment_std", "n_reports"]
daily = daily.sort_values(["available_date", "symbol"])
DAILY_PATH = OUT_DIR / "research_report_daily_2024.csv"
daily.to_csv(DAILY_PATH, index=False)
print(f"全年聚合: {daily.shape[0]} 行 (交易日 x 个股) -> {DAILY_PATH}")
print("每日每股票研报数分布(n_reports):")
print(daily["n_reports"].describe().round(3).to_string())
daily.head()

---

## 附加：report_type / reliability 交叉分析（探索性）

研报结构化字段（`report_type` 为代码、`reliability` 疑似评级/可信度，均**未确认语义**）与
`class_1_prob` 的交叉，仅作探索性观察，不据此下因果/语义结论。

In [ ]:
# report_type / reliability 与 class_1_prob 的交叉(探索性, 语义未确认)
print("report_type x class_1_prob:")
print(res.groupby("report_type")["class_1_prob"].agg(["mean", "median", "std", "count"]).round(4).to_string())
print()
print("reliability 是否缺失 x class_1_prob:")
print(res.assign(has_rel=res["reliability"].notna()).groupby("has_rel")["class_1_prob"]
      .agg(["mean", "median", "count"]).round(4).to_string())
print()
print("reliability (非缺失值) x class_1_prob:")
print(res[res["reliability"].notna()].groupby("reliability")["class_1_prob"]
      .agg(["mean", "median", "std", "count"]).round(4).to_string())
print()
print("reliability 缺失 与 report_type 的关联(交叉表):")
print(pd.crosstab(res["report_type"], res["reliability"].notna(),
                  rownames=["report_type"], colnames=["rel_present"]).to_string())

---

## 结论与下一步

- 2024 全年研报推理结果已在本册完成：**逐月概况 / 概率分布 / 日度-个股聚合(`artifacts/research_report_daily_2024.csv`) / report_type·reliability 交叉 / 可视化**。
- 数据要点：共 27.3 万篇研报、3422 只个股、94 家机构；`reliability` 缺失 84.7%，且几乎只在 `report_type=21` 中有值。
- 已知限制：标签语义未确认、pooling 未确认（默认 `cls`）、fp16 概率与 fp32 有小幅数值差异。
- 下一步候选：
  1. 用 `reports.parquet` 对齐 `report_fy_labels.parquet` 与 `report_confirmation_labels.parquet`，验证单篇研报未来信息；
  2. 结合 `report_type/reliability/organ_name` 做机构分组下的信号稳定性检查；
  3. 与 `social_text_daily_2024.csv` 的日度情绪分数做交叉对比。